# Quantum Canary

Check your qubits before a job, between jobs (as often as you like), and after.

Install: `pip install quantum-canary` (from a clone of this repository: `pip install -e .`).

In [1]:
# %pip install -e . 

In [2]:
import warnings
warnings.filterwarnings("ignore")

import quantum_canary as canary

## Backend

This uses `fake_fez`, a local copy of IBM's 156-qubit Heron device with its real calibration and noise. It needs no account. Set `USE_IBM = True` to use your saved IBM account instead.

In [3]:
USE_IBM = False

if USE_IBM:
    from qiskit_ibm_runtime import QiskitRuntimeService
    backend = QiskitRuntimeService().backend("ibm_kingston")
else:
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    backend = FakeProviderForBackendV2().backend("fake_fez")
backend.name

'fake_fez'

## 1. Before your job

One short job measures T1, T2, T2\*, the frequency offset and the gate error of each qubit, with error bars, a fit check (chi2/dof) and a health verdict with its reason.

In [4]:
before = canary.check(backend, [13, 14])
before.print()
before.save("before.json")

qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
13    253 ± 12 us      218 ± 21 us      206 ± 13 us      +0.0443 ± 0.052  1.72e-04 ± 1e-05    ok                        GREEN
14    235 ± 11 us      51.3 ± 3.5 us    48 ± 2.9 us      -0.198 ± 0.24    1.53e-04 ± 1e-05    ok                        GREEN

dynamical decoupling: not needed on these qubits

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does, that probe's
        numbers may be wrong.
Health  GREEN   every fit converged and passed, and T2* is within 20% of T2
        YELLOW  a fit failed its limit, or T2* differs from T2 by more than 20%
                (slow frequency noise, which dynamical decoupling refocuses)
        RED

In [5]:
q = before[13]
q.t1, q.t2, q.t2_star, q.health, q.fit

(0.0002530974497996835,
 0.00021832912408818747,
 0.0002056974826631627,
 'GREEN',
 [])

## 2. Between jobs (as often as you like)

The first `check()` is the baseline; call it again between your jobs. Each later check is compared with the baseline, with the |z| limit raised for the number of checks.

In [6]:
mon = canary.Monitor(backend, [13, 14])
mon.check()   # baseline
# ... your job ...
mon.check()   # between jobs
mon.print()
mon.changed

1 check against the baseline: changed = |z| > 2.00, so that the chance of any false alarm over all of them is that of one check at |z| > 2

check 1 (2026-10-16T23:18:17)
qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    GREEN    GREEN        -2.10    -0.13    -1.10    +0.75   -
14    GREEN    GREEN        -2.63    -0.23    +0.48    +2.25   T1

changed = |z| > 2.00 after a Holm correction over each qubit's four parameters; z uses both checks' error bars
changed qubits: [14]; results that depend on them may not hold


[14]

## 3. After your job

Check again and compare with the first check.

In [7]:
after = canary.check(backend, [13, 14])
drift = canary.compare(canary.load("before.json"), after)
drift.print()
drift.changed

qubit before   after         z T1     z T2     z dw    z eps   changed
----------------------------------------------------------------------
13    GREEN    GREEN        +0.10    -1.09    -0.43    +1.04   -
14    GREEN    GREEN        +0.70    -2.08    +0.20    +0.45   -

changed = |z| > 2.00 after a Holm correction over each qubit's four parameters; z uses both checks' error bars
stable: nothing changed beyond the error bars


[]

## No hardware at all

Simulated qubits, with their true values kept for comparison.

In [8]:
sim = canary.simulate(2, regime="ideal")
sim.print()
[(q.t1, q.truth["T1_s"]) for q in sim]

qubit T1               T2               T2*              dw/2pi (kHz)     eps                 fit                       health
------------------------------------------------------------------------------------------------------------------------------
0     190 ± 9.1 us     98 ± 8.8 us      83.4 ± 5.6 us    -4.15 ± 0.16     3.07e-04 ± 2e-05    ok                        GREEN
1     114 ± 5.4 us     91.6 ± 6.7 us    98.8 ± 6.5 us    +6.14 ± 0.12     5.01e-04 ± 3e-05    ok                        GREEN

dynamical decoupling: not needed on these qubits
frequency offset on qubit 0: |Δω|·T2* = 2.2 rad > 1; correct it with a virtual-Z frame update or frequency recalibration (DD only if it drifts)
frequency offset on qubit 1: |Δω|·T2* = 3.8 rad > 1; correct it with a virtual-Z frame update or frequency recalibration (DD only if it drifts)

Fit     chi2/dof of each probe against its limit. On a qubit that follows the model
        each probe goes over its limit 1.25% of the time; when it does,

[(0.00019043314863678617, 0.00018899296052325023),
 (0.00011391130105392075, 0.00011799078236726674)]